## 09.04 双向循环神经网络


### 环境配置


In [1]:
import os
import sys
sys.path.insert(0, "..")
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    import pypto
    import torch
    from torch import nn
    from torch.nn import functional as F
    import torch_npu
    import logging

warnings.filterwarnings("ignore", message="Permission mismatch")
warnings.filterwarnings("ignore", message="TASK_QUEUE_ENABLE")
warnings.filterwarnings("ignore", message="Cannot create tensor")
logging.getLogger("torch_npu").setLevel(logging.WARNING)
logging.getLogger('matplotlib').setLevel(logging.WARNING)
import matplotlib.pyplot as plt

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
pypto.pypto_impl.DeviceInit()

### 练习 9.4.1

**题目：** 如果不同方向使用不同数量的隐藏单元，$\mathbf{H}_t$ 的形状会发生怎样的变化？

**解答：** 在双向 RNN 中，通常前向和反向隐藏单元数相同，$H_t$ 维度直接翻倍。当不同方向使用不同隐藏单元数时（分别设为 $h_f$ 和 $h_b$），需要将前向和反向的隐藏状态**拼接**(concat)：$\dim(H_t) = h_f + h_b$。

以下使用 `torch` 编程验证：



In [2]:
import torch
import torch.nn as nn

class BiLSTM(nn.Module):
    def __init__(self, input_size, hidden_size_forward, hidden_size_backward):
        super().__init__()
        self.lstm_forward = nn.LSTM(input_size, hidden_size_forward, batch_first=True)
        self.lstm_backward = nn.LSTM(input_size, hidden_size_backward, batch_first=True)
    def forward(self, x, x_lengths):
        packed_x_forward = nn.utils.rnn.pack_padded_sequence(x, x_lengths, batch_first=True, enforce_sorted=False)
        packed_h_forward, _ = self.lstm_forward(packed_x_forward)
        h_forward, _ = nn.utils.rnn.pad_packed_sequence(packed_h_forward, batch_first=True)
        packed_x_backward = nn.utils.rnn.pack_padded_sequence(torch.flip(x, [1]), x_lengths, batch_first=True, enforce_sorted=False)
        packed_h_backward, _ = self.lstm_backward(packed_x_backward)
        h_backward, _ = nn.utils.rnn.pad_packed_sequence(packed_h_backward, batch_first=True)
        h_backward = torch.flip(h_backward, [1])
        h = torch.cat((h_forward, h_backward), dim=2)
        return h

batch_size, seq_length, input_size = 3, 5, 2
hidden_size_forward, hidden_size_backward = 10, 8
x_lengths = [3, 5, 2]
x = [torch.randn(x_lengths[i], input_size) for i in range(batch_size)]
x = nn.utils.rnn.pad_sequence(x, batch_first=True)
model = BiLSTM(input_size, hidden_size_forward, hidden_size_backward)
h = model(x, x_lengths)
print(f'Output shape: {h.shape}')  # torch.Size([3, 5, 18])

使用 `PyPTO` 编程：


In [3]:
from src.pypto_ops import PyPTOLSTM


class BiLSTMPyPTO(nn.Module):
    """双向 LSTM（PyPTO 版）：前向/反向使用不同数量的隐藏单元。

    PyPTOLSTM 为单向（默认单层、支持多层），双向通过两个 PyPTOLSTM 拼接实现：
    前向隐层处理正序序列，反向隐层处理逆序序列，最后沿隐藏维拼接，
    输出维度为 h_f + h_b。
    """

    def __init__(self, input_size, hidden_size_forward, hidden_size_backward):
        super().__init__()
        self.lstm_forward = PyPTOLSTM(input_size, hidden_size_forward)
        self.lstm_backward = PyPTOLSTM(input_size, hidden_size_backward)

    def forward(self, x):
        # x: (batch, seq, input_size)
        h_forward, _ = self.lstm_forward(x.permute(1, 0, 2).contiguous())
        h_backward, _ = self.lstm_backward(
            torch.flip(x, [1]).permute(1, 0, 2).contiguous())
        h_backward = torch.flip(h_backward.permute(1, 0, 2), [1])  # 对齐时间步
        h = torch.cat((h_forward.permute(1, 0, 2), h_backward), dim=2)
        return h


batch_size, seq_length, input_size = 3, 5, 2
hidden_size_forward, hidden_size_backward = 10, 8
model = BiLSTMPyPTO(input_size, hidden_size_forward, hidden_size_backward).to(device)
x = torch.randn(batch_size, seq_length, input_size, device=device)
h = model(x)
print(f'Output shape: {h.shape}')  # (3, 5, 18) = h_f(10) + h_b(8)

Output shape: torch.Size([3, 5, 18])


### 练习 9.4.2

**题目：** 设计一个具有多个隐藏层的双向循环神经网络。

**解答：** 直接使用 `nn.LSTM(..., num_layers=4, bidirectional=True)` 即可构建多层双向 LSTM，最终线性层输入维度需乘以 2（双向）。

以下使用 `torch` 编程：



In [4]:
from src.utils import load_data_time_machine, try_gpu, RNNModel
from torch import nn

batch_size, num_steps = 32, 35
train_iter, vocab = load_data_time_machine(batch_size, num_steps)

vocab_size, num_hiddens, num_layers = len(vocab), 256, 4
lstm_layer = nn.LSTM(vocab_size, num_hiddens, num_layers, bidirectional=True)
model = RNNModel(lstm_layer, len(vocab)).to(try_gpu())
print(model)

使用 `PyPTO` 编程：



In [5]:
from src.utils import load_data_time_machine, train_ch8
from src.pypto_ops import PyPTOLinear, loss_fn

batch_size, num_steps = 32, 35
train_iter, vocab = load_data_time_machine(batch_size, num_steps)

class PyPTOBiLSTMModel(nn.Module):
    """多层双向 LSTM 语言模型：host 算子 nn.LSTM 在 NPU 上执行，
    输出层使用 PyPTOLinear（输入维度 hidden_size * 2，双向）。"""

    def __init__(self, lstm_layer, vocab_size):
        super().__init__()
        self.lstm = lstm_layer
        self.linear = PyPTOLinear(lstm_layer.hidden_size * 2, vocab_size)  # bidirectional * 2
        self.vocab_size = vocab_size
    def forward(self, inputs, state):
        X = F.one_hot(inputs.T.long(), self.vocab_size).float()
        Y, state = self.lstm(X, state)
        Y_npu = Y.reshape((-1, Y.shape[-1])).to(self.linear.weight.device)
        return self.linear(Y_npu), state
    def begin_state(self, batch_size, device):
        num_d = self.lstm.num_layers * 2  # bidirectional
        return (torch.zeros((num_d, batch_size, self.lstm.hidden_size), device=device),
                torch.zeros((num_d, batch_size, self.lstm.hidden_size), device=device))

lstm_layer = nn.LSTM(len(vocab), 256, 4, bidirectional=True).to(device)
net = PyPTOBiLSTMModel(lstm_layer, len(vocab))
net.linear = net.linear.to(device)
X_w, y_w = next(iter(train_iter))
s_w = net.begin_state(batch_size, device)
l_w = loss_fn(net(X_w.to(device), s_w)[0], y_w.T.reshape(-1).to(device), len(vocab))
l_w.backward(); net.zero_grad()
train_ch8(net, train_iter, vocab, 1, 500, device, use_plot=False, verbose=False,
          loss_fn=lambda y_h, y: loss_fn(y_h, y, len(vocab)))

困惑度 1.1, 58854.6 词元/秒 npu:0


time travellerererererererererererererererererererererererererer
travellerererererererererererererererererererererererererer


### 练习 9.4.3

**题目：** 如何设计神经网络模型处理一词多义？哪种架构更适合？

**解答：** 处理一词多义需要结合上下文信息。双向循环神经网络（如 BiLSTM）或 Transformer 等上下文感知模型能够根据上下文动态生成单词的向量表示，更好地捕捉单词在不同上下文中的含义。这类模型可以学到上下文中单词的复杂交互。



---
## 参考答案来源
参考答案和 PyTorch 代码实现来源：[https://datawhalechina.github.io/d2l-ai-solutions-manual/#/](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/)

